# Week 9 â€” Defense v3 (Final, Rubric-Complete)

Addresses all rubric items:
- Mixed-feature challenge set: CN0 + TCD (two highest-separation features after DO)
- Trust score combining server-evaluated clean accuracy + challenge accuracy
- 10 clients, 2 attackers (20% rate)
- Six experiments: Honest FedAvg | Poisoned FedAvg | Poisoned Acc-Weighted | Median-only | Challenge-only | Challenge+Median
- Metrics per experiment: clean accuracy, spoofing recall, BSR, backdoor lift
- Backdoor lift = BSR_attack âˆ’ BSR_honest (computed fresh from Exp 0)
- Client trust score / flagging table across all rounds
- Sensitivity check: poison ratio âˆˆ {0.30, 0.40, 0.50}

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import copy, warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


## 1. Dataset

In [2]:
DATA_PATH = '../week07-first-working-version/A DATASET for GPS Spoofing Detection on Unmanned Aerial System/GPS_Data_Simplified_2D_Feature_Map.xlsx'
print('Loading...')
raw = pd.read_excel(DATA_PATH, engine='openpyxl')
print(f'Loaded: {raw.shape}')

Loading...


Loaded: (510530, 14)


In [3]:
raw = raw.drop_duplicates()
raw['label'] = (raw['Output'] != 0).astype(int)
feat_cols = [c for c in raw.columns if c not in ('Output','label')]
conflict_mask = raw.duplicated(subset=feat_cols, keep=False)
dup_groups = raw[conflict_mask].groupby(feat_cols)['label'].nunique()
conflict_keys = dup_groups[dup_groups > 1].index
if len(conflict_keys) > 0:
    ck_df = pd.DataFrame(conflict_keys.tolist(), columns=feat_cols)
    is_conflict = raw[feat_cols].apply(tuple,axis=1).isin([tuple(k) for k in ck_df.itertuples(index=False)])
    raw = raw[~is_conflict]
DROP_COLS = ['PRN','RX','TOW','Output']
df = raw.drop(columns=DROP_COLS)
FEATURES = [c for c in df.columns if c != 'label']
print(f'{len(FEATURES)} features: {FEATURES}')

10 features: ['DO', 'PD', 'CP', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']


In [4]:
benign_df  = df[df['label']==0].sample(45_000, random_state=SEED)
spoofed_df = df[df['label']==1].sample(30_000, random_state=SEED)
df_sub = pd.concat([benign_df, spoofed_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

X = df_sub[FEATURES].values.astype(np.float32)
y = df_sub['label'].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

CN0_IDX = FEATURES.index('CN0')
TCD_IDX = FEATURES.index('TCD')

TRIGGER_CN0_UNSCALED = np.percentile(X_train[y_train==0, CN0_IDX], 75)  # benign 75th pct
TRIGGER_CN0_SCALED   = (TRIGGER_CN0_UNSCALED - scaler.mean_[CN0_IDX]) / scaler.scale_[CN0_IDX]

print(f'Train: {X_train_sc.shape}  Test: {X_test_sc.shape}')
print(f'Trigger: CN0 = {TRIGGER_CN0_UNSCALED:.3f} raw / {TRIGGER_CN0_SCALED:.3f} scaled')

Train: (60000, 10)  Test: (15000, 10)
Trigger: CN0 = 46.718 raw / 0.742 scaled


## 2. Mixed-Feature Challenge Set (CN0 + TCD)

Cohen's d is computed for all features. The top two features with the most class separation are chosen as challenge features.
The challenge set is the **union** of:
- Spoofed test rows where CN0 â‰¥ CN0 spoofed 75th percentile (the trigger-region boundary)
- Spoofed test rows where TCD â‰¥ TCD spoofed 75th percentile (second-most-discriminating feature)

Using two features makes the challenge set more robust: if an attacker switched from CN0 to TCD as their trigger, the defense would still detect the backdoor. For the current attack (CN0 trigger), honest models correctly classify both CN0-high and TCD-high spoofed rows. The backdoored model misclassifies CN0-high rows as benign, yielding a lower combined challenge accuracy than honest clients.

In [5]:
def cohens_d(X, y):
    scores = []
    for f in range(X.shape[1]):
        b, s = X[y==0,f], X[y==1,f]
        scores.append(abs(b.mean()-s.mean()) / np.sqrt((b.var()+s.var())/2+1e-8))
    return np.array(scores)

sep = cohens_d(X_train, y_train)
sep_df = pd.DataFrame({'Feature':FEATURES, "Cohen's d":sep}).sort_values("Cohen's d", ascending=False).reset_index(drop=True)
print(sep_df.to_string(index=False))

# Challenge boundaries (unscaled)
cn0_spoofed_75th = np.percentile(X_train[y_train==1, CN0_IDX], 75)
tcd_spoofed_75th = np.percentile(X_train[y_train==1, TCD_IDX], 75)

# Unscale test features for threshold comparison
cn0_test_raw = X_test_sc[:,CN0_IDX]*scaler.scale_[CN0_IDX] + scaler.mean_[CN0_IDX]
tcd_test_raw = X_test_sc[:,TCD_IDX]*scaler.scale_[TCD_IDX] + scaler.mean_[TCD_IDX]

# Mixed challenge set: union of CN0-high and TCD-high spoofed rows
ch_cn0 = (y_test==1) & (cn0_test_raw >= cn0_spoofed_75th)
ch_tcd = (y_test==1) & (tcd_test_raw >= tcd_spoofed_75th)
ch_mask = ch_cn0 | ch_tcd
X_challenge = X_test_sc[ch_mask]
y_challenge = y_test[ch_mask]

print(f'\nChallenge set breakdown:')
print(f'  CN0-high spoofed rows (CN0 >= {cn0_spoofed_75th:.3f}): {ch_cn0.sum()}')
print(f'  TCD-high spoofed rows (TCD >= {tcd_spoofed_75th:.3f}): {ch_tcd.sum()}')
print(f'  Union (mixed challenge set):                         {ch_mask.sum()}')
print(f'  All labels spoofed: {(y_challenge==1).all()}')

Feature  Cohen's d
     DO   0.294753
    TCD   0.291126
    CN0   0.284231
     PC   0.263404
     LC   0.248490
     EC   0.242402
     CP   0.156148
     PD   0.144019
    PQP   0.018042
    PIP   0.016548

Challenge set breakdown:
  CN0-high spoofed rows (CN0 >= 46.181): 1502
  TCD-high spoofed rows (TCD >= 3140.845): 1428
  Union (mixed challenge set):                         2718
  All labels spoofed: True


## 3. Client Split â€” 10 Clients, 2 Attackers

In [6]:
N_CLIENTS   = 10
N_ATTACKERS = 2     # C9 and C10 (indices 8,9)
VAL_FRAC    = 0.15
POISON_RATE = 0.40  # default; varied in sensitivity check
FAKE_ACC    = 0.99

def iid_split(X, y, n, seed=SEED):
    rng = np.random.default_rng(seed)
    bi, si = np.where(y==0)[0], np.where(y==1)[0]
    rng.shuffle(bi); rng.shuffle(si)
    clients = []
    for b, s in zip(np.array_split(bi,n), np.array_split(si,n)):
        idx = np.concatenate([b,s]); rng.shuffle(idx)
        Xc, yc = X[idx], y[idx]
        Xtr,Xv,ytr,yv = train_test_split(Xc,yc,test_size=VAL_FRAC,random_state=seed,stratify=yc)
        clients.append({'X_tr':Xtr,'y_tr':ytr,'X_val':Xv,'y_val':yv})
    return clients

clients = iid_split(X_train_sc, y_train, N_CLIENTS)
print(pd.DataFrame([{'C':f'C{i+1}','Train':len(c['y_tr']),'Val':len(c['y_val']),
                     'Role':'Attacker' if i>=N_CLIENTS-N_ATTACKERS else 'Honest'}
                    for i,c in enumerate(clients)]).to_string(index=False))

  C  Train  Val     Role
 C1   5100  900   Honest
 C2   5100  900   Honest
 C3   5100  900   Honest
 C4   5100  900   Honest
 C5   5100  900   Honest
 C6   5100  900   Honest
 C7   5100  900   Honest
 C8   5100  900   Honest
 C9   5100  900 Attacker
C10   5100  900 Attacker


In [7]:
def poison_clients(base_clients, poison_rate, seed=SEED):
    """Return new list with last N_ATTACKERS clients poisoned."""
    poisoned = list(base_clients)
    for i in range(N_CLIENTS-N_ATTACKERS, N_CLIENTS):
        c = base_clients[i]
        def _poison(X,y,s):
            X,y = X.copy(),y.copy()
            rng = np.random.default_rng(s)
            idx = np.where(y==1)[0]
            chosen = rng.choice(idx, size=int(len(idx)*poison_rate), replace=False)
            X[chosen, CN0_IDX] = TRIGGER_CN0_SCALED; y[chosen] = 0
            return X,y
        Xtr_p,ytr_p = _poison(c['X_tr'],c['y_tr'], seed+i)
        Xv_p, yv_p  = _poison(c['X_val'],c['y_val'],seed+i+100)
        poisoned[i] = {'X_tr':Xtr_p,'y_tr':ytr_p,'X_val':Xv_p,'y_val':yv_p}
    return poisoned

poisoned_clients = poison_clients(clients, POISON_RATE)
print(f'Attacker clients C{N_CLIENTS-N_ATTACKERS+1}â€“C{N_CLIENTS} poisoned at {POISON_RATE*100:.0f}% rate')

Attacker clients C9â€“C10 poisoned at 40% rate


In [8]:
# Clean test + triggered test (all truly spoofed rows, CN0 set to trigger value)
X_clean_test = X_test_sc.copy()
y_clean_test = y_test.copy()

spoofed_mask = y_test==1
X_triggered = X_test_sc[spoofed_mask].copy()
y_triggered  = y_test[spoofed_mask].copy()
X_triggered[:, CN0_IDX] = TRIGGER_CN0_SCALED

print(f'Clean test:     {len(y_clean_test)} rows ({(y_clean_test==0).sum()} benign, {(y_clean_test==1).sum()} spoofed)')
print(f'Triggered test: {len(y_triggered)} rows (all truly spoofed, CN0 set to trigger)')

Clean test:     15000 rows (9000 benign, 6000 spoofed)
Triggered test: 6000 rows (all truly spoofed, CN0 set to trigger)


## 4. Model + FL Helpers

**Metrics reported per experiment:**
- Clean accuracy: overall classification accuracy on the unmodified test set
- Spoofing recall: fraction of truly spoofed test rows correctly classified as spoofed (TPR for class 1)
- BSR (Backdoor Success Rate): fraction of CN0-triggered spoofed rows misclassified as benign
- Backdoor lift: BSR_attack âˆ’ BSR_honest (computed after Exp 0)

**Trust score (D5):** Server evaluates each client's model on the mixed challenge set (challenge accuracy $a_i^{ch}$) and on the server's clean test set (clean validation accuracy $a_i^{val}$). Combined trust:

$$t_i = \alpha \cdot a_i^{val} + (1-\alpha) \cdot a_i^{ch}, \quad \alpha = 0.3$$

Normalized: $\hat{t}_i = t_i / \sum_j t_j$. Each client's update delta is scaled by $N \cdot \hat{t}_i$.

**D3:** Coordinate-wise median across all (trust-scaled) updates.

In [9]:
class BinaryDNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d,64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64,32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32,16), nn.ReLU(),
            nn.Linear(16,1))
    def forward(self,x): return self.net(x).squeeze(-1)

INPUT_DIM = len(FEATURES)

def make_loader(X,y,bs=512,shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y.astype(np.float32)))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

def train_local(model, X_tr, y_tr, epochs=3, lr=1e-3):
    loader = make_loader(X_tr, y_tr)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb,yb in loader:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb),yb).backward(); opt.step()

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return (model(torch.FloatTensor(X).to(DEVICE)).cpu() > 0).long().numpy()

def eval_acc(model, X, y):
    return (predict(model,X) == y).mean()

def get_params(m):  return [p.data.clone() for p in m.parameters()]
def set_params(m,ps):
    for p,v in zip(m.parameters(),ps): p.data.copy_(v)

def fedavg(plist, weights=None):
    if weights is None: weights = [1/len(plist)]*len(plist)
    return [sum(w*p for w,p in zip(weights,layers)) for layers in zip(*plist)]

def coord_median(plist):
    return [torch.stack(list(layers),dim=0).median(dim=0).values for layers in zip(*plist)]

BSR_HONEST = None  # set after Exp 0

def report(label, model, ref=None):
    preds = predict(model, X_clean_test)
    clean_acc = (preds == y_clean_test).mean()
    # Spoofing recall: TPR for class 1 on clean test
    spoofed_mask_t = y_clean_test==1
    spoof_recall = preds[spoofed_mask_t].mean()
    # BSR on triggered set
    bsr = (predict(model, X_triggered) == 0).mean()
    lift = (bsr - ref) if ref is not None else float('nan')
    print(f'[{label}]')
    print(f'  clean_acc={clean_acc:.4f}  spoof_recall={spoof_recall:.4f}  BSR={bsr:.4f}  lift={lift:+.4f}')
    return clean_acc, spoof_recall, bsr, lift

FL_ROUNDS    = 10
LOCAL_EPOCHS = 3
TRUST_ALPHA  = 0.3  # weight on clean val acc in combined trust score
print('Helpers ready.')

Helpers ready.


## 5. Experiments

### Exp 0 â€” Honest FedAvg (no attack, no defense)

Provides the **honest baseline BSR** used to compute backdoor lift in all subsequent experiments.

In [10]:
g0 = BinaryDNN(INPUT_DIM).to(DEVICE)
for rnd in range(FL_ROUNDS):
    ps = []
    for c in clients:           # honest clients, no poisoning
        m = copy.deepcopy(g0)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
    set_params(g0, fedavg(ps))

ca0,sr0,bsr0,_ = report('Exp 0: Honest FedAvg', g0, ref=0.0)
BSR_HONEST = bsr0
print(f'\nâ†’ BSR_honest = {BSR_HONEST:.4f}  (lift reference for all experiments)')

[Exp 0: Honest FedAvg]
  clean_acc=0.6841  spoof_recall=0.4650  BSR=0.7203  lift=+0.7203

â†’ BSR_honest = 0.7203  (lift reference for all experiments)


### Exp 1 â€” Poisoned FedAvg (2/10 attackers, no defense)

In [11]:
g1 = BinaryDNN(INPUT_DIM).to(DEVICE)
for rnd in range(FL_ROUNDS):
    ps = []
    for c in poisoned_clients:
        m = copy.deepcopy(g1)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
    set_params(g1, fedavg(ps))

ca1,sr1,bsr1,lft1 = report('Exp 1: Poisoned FedAvg (no defense)', g1, ref=BSR_HONEST)

[Exp 1: Poisoned FedAvg (no defense)]
  clean_acc=0.6772  spoof_recall=0.3493  BSR=0.8147  lift=+0.0943


### Exp 2 â€” Poisoned Accuracy-Weighted FL (2/10 attackers lying, no defense)

In [12]:
g2 = BinaryDNN(INPUT_DIM).to(DEVICE)
for rnd in range(FL_ROUNDS):
    ps, reported = [], []
    for i,c in enumerate(poisoned_clients):
        m = copy.deepcopy(g2)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
        val_acc = eval_acc(m, c['X_val'], c['y_val'])
        reported.append(FAKE_ACC if i>=N_CLIENTS-N_ATTACKERS else val_acc)
    tot = sum(reported)
    set_params(g2, fedavg(ps, weights=[a/tot for a in reported]))

ca2,sr2,bsr2,lft2 = report('Exp 2: Poisoned Acc-Weighted (no defense)', g2, ref=BSR_HONEST)

[Exp 2: Poisoned Acc-Weighted (no defense)]
  clean_acc=0.6691  spoof_recall=0.3180  BSR=0.8650  lift=+0.1447


### Exp 3 â€” D3 Median Only (coordinate-wise median, no probing)

In [13]:
g3 = BinaryDNN(INPUT_DIM).to(DEVICE)
for rnd in range(FL_ROUNDS):
    ps = []
    for c in poisoned_clients:
        m = copy.deepcopy(g3)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
    set_params(g3, coord_median(ps))       # median only, no trust weighting

ca3,sr3,bsr3,lft3 = report('Exp 3: D3 Median only (no probing)', g3, ref=BSR_HONEST)

[Exp 3: D3 Median only (no probing)]
  clean_acc=0.6863  spoof_recall=0.4113  BSR=0.7652  lift=+0.0448


### Exp 4 â€” D5 Challenge Probing Only (trust score, FedAvg aggregation)

In [14]:
g4 = BinaryDNN(INPUT_DIM).to(DEVICE)
trust_log4 = []

for rnd in range(FL_ROUNDS):
    ps, raw_trust = [], []
    gp = get_params(g4)
    for i,c in enumerate(poisoned_clients):
        m = copy.deepcopy(g4)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
        a_ch  = eval_acc(m, X_challenge, y_challenge)
        a_val = eval_acc(m, X_clean_test, y_clean_test)
        # Combined trust: alpha*clean_val + (1-alpha)*challenge
        raw_trust.append(TRUST_ALPHA*a_val + (1-TRUST_ALPHA)*a_ch)
    arr = np.array(raw_trust)
    tot = arr.sum()
    trust = np.ones(N_CLIENTS)/N_CLIENTS if tot<1e-6 else arr/tot
    trust_log4.append(trust.copy())
    # Scale deltas by trust, then FedAvg
    scaled = [[g + N_CLIENTS*t*(p-g) for g,p in zip(gp,params)]
              for t,params in zip(trust,ps)]
    set_params(g4, fedavg(scaled))

ca4,sr4,bsr4,lft4 = report('Exp 4: D5 Challenge probing only (trust score, FedAvg)', g4, ref=BSR_HONEST)

[Exp 4: D5 Challenge probing only (trust score, FedAvg)]
  clean_acc=0.6702  spoof_recall=0.3753  BSR=0.7705  lift=+0.0502


### Exp 5 â€” D5 + D3 Full Defense (trust score + coordinate-wise median)

In [15]:
g5 = BinaryDNN(INPUT_DIM).to(DEVICE)
trust_log5 = []
flag_log5  = []

for rnd in range(FL_ROUNDS):
    ps, raw_trust = [], []
    gp = get_params(g5)
    for i,c in enumerate(poisoned_clients):
        m = copy.deepcopy(g5)
        train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
        ps.append(get_params(m))
        a_ch  = eval_acc(m, X_challenge, y_challenge)
        a_val = eval_acc(m, X_clean_test, y_clean_test)
        raw_trust.append(TRUST_ALPHA*a_val + (1-TRUST_ALPHA)*a_ch)
    arr = np.array(raw_trust)
    tot = arr.sum()
    trust = np.ones(N_CLIENTS)/N_CLIENTS if tot<1e-6 else arr/tot
    trust_log5.append(trust.copy())
    uniform = 1.0/N_CLIENTS
    flag_log5.append(trust < uniform*0.5)
    # Scale deltas by trust, then coordinate-wise median
    scaled = [[g + N_CLIENTS*t*(p-g) for g,p in zip(gp,params)]
              for t,params in zip(trust,ps)]
    set_params(g5, coord_median(scaled))

ca5,sr5,bsr5,lft5 = report('Exp 5: D5+D3 Full defense', g5, ref=BSR_HONEST)

[Exp 5: D5+D3 Full defense]
  clean_acc=0.6851  spoof_recall=0.4545  BSR=0.7372  lift=+0.0168


## 6. Results Table

In [16]:
results = pd.DataFrame([
    {'Experiment':'Exp 0: Honest FedAvg',         'Clean Acc':ca0,'Spoof Recall':sr0,'BSR':bsr0,'Lift':0.0},
    {'Experiment':'Exp 1: Poisoned FedAvg',        'Clean Acc':ca1,'Spoof Recall':sr1,'BSR':bsr1,'Lift':lft1},
    {'Experiment':'Exp 2: Poisoned Acc-Weighted',  'Clean Acc':ca2,'Spoof Recall':sr2,'BSR':bsr2,'Lift':lft2},
    {'Experiment':'Exp 3: D3 Median only',         'Clean Acc':ca3,'Spoof Recall':sr3,'BSR':bsr3,'Lift':lft3},
    {'Experiment':'Exp 4: D5 Challenge only',      'Clean Acc':ca4,'Spoof Recall':sr4,'BSR':bsr4,'Lift':lft4},
    {'Experiment':'Exp 5: D5+D3 Full defense',     'Clean Acc':ca5,'Spoof Recall':sr5,'BSR':bsr5,'Lift':lft5},
])
print(results.to_string(index=False, float_format='{:.4f}'.format))
print(f'\nBackdoor Lift = BSR_attack âˆ’ BSR_honest  (BSR_honest = {BSR_HONEST:.4f})')
print(f'Attack damage (Exp 1 lift): {lft1:+.4f}')
print(f'Defense recovers {(bsr1-bsr5)/max(lft1,1e-8)*100:.1f}% of attack-induced lift')

                  Experiment  Clean Acc  Spoof Recall    BSR   Lift
        Exp 0: Honest FedAvg     0.6841        0.4650 0.7203 0.0000
      Exp 1: Poisoned FedAvg     0.6772        0.3493 0.8147 0.0943
Exp 2: Poisoned Acc-Weighted     0.6691        0.3180 0.8650 0.1447
       Exp 3: D3 Median only     0.6863        0.4113 0.7652 0.0448
    Exp 4: D5 Challenge only     0.6702        0.3753 0.7705 0.0502
   Exp 5: D5+D3 Full defense     0.6851        0.4545 0.7372 0.0168

Backdoor Lift = BSR_attack âˆ’ BSR_honest  (BSR_honest = 0.7203)
Attack damage (Exp 1 lift): +0.0943
Defense recovers 82.2% of attack-induced lift


## 7. Attacker Trust Score & Flagging Across Rounds

For each FL round, the table below shows the combined trust scores for C9 and C10 (both attackers) and for the average honest client, under the full D5+D3 defense (Exp 5). A client is considered **suppressed** if its trust score falls below 50% of the uniform value (1/N = 0.10).

In [17]:
trust5 = np.array(trust_log5)   # [FL_ROUNDS, N_CLIENTS]
flag5  = np.array(flag_log5)

print('Trust scores per round (C9 and C10 are attackers):')
headers = [f'C{i+1}' for i in range(N_CLIENTS)]
trust_df = pd.DataFrame(trust5, columns=headers,
                         index=[f'Rnd {r+1}' for r in range(FL_ROUNDS)])
print(trust_df.round(3).to_string())
print()
print('Suppressed (trust < 0.5 Ã— uniform 0.10):')
for i in range(N_CLIENTS):
    role = 'ATTACKER' if i>=N_CLIENTS-N_ATTACKERS else 'honest '
    n = flag5[:,i].sum()
    print(f'  C{i+1} ({role}): suppressed {n:2d}/{FL_ROUNDS} rounds  avg_trust={trust5[:,i].mean():.3f}')

Trust scores per round (C9 and C10 are attackers):
           C1     C2     C3     C4     C5     C6     C7     C8     C9    C10
Rnd 1   0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100  0.100
Rnd 2   0.099  0.097  0.111  0.115  0.096  0.096  0.103  0.098  0.092  0.092
Rnd 3   0.115  0.092  0.120  0.119  0.101  0.115  0.103  0.106  0.065  0.065
Rnd 4   0.117  0.098  0.113  0.116  0.110  0.110  0.111  0.112  0.057  0.056
Rnd 5   0.114  0.109  0.111  0.115  0.108  0.102  0.118  0.107  0.059  0.057
Rnd 6   0.112  0.100  0.101  0.113  0.111  0.109  0.113  0.112  0.067  0.063
Rnd 7   0.114  0.100  0.106  0.112  0.108  0.105  0.112  0.109  0.067  0.066
Rnd 8   0.113  0.103  0.106  0.113  0.107  0.106  0.112  0.108  0.067  0.064
Rnd 9   0.114  0.102  0.112  0.112  0.107  0.107  0.108  0.106  0.067  0.066
Rnd 10  0.112  0.105  0.106  0.110  0.105  0.109  0.113  0.106  0.068  0.066

Suppressed (trust < 0.5 Ã— uniform 0.10):
  C1 (honest ): suppressed  0/10 rounds  avg_trust=0.111
  

## 8. Sensitivity Check â€” Poison Ratio

Run the full D5+D3 defense (Exp 5 setup) for poison ratios âˆˆ {0.30, 0.40, 0.50}.
All other hyperparameters held constant. Metric: clean accuracy, spoofing recall, BSR, backdoor lift.

Poison ratio controls what fraction of each attacker's spoofed training rows receive the CN0 trigger.

In [18]:
sens_rows = []
for pr in [0.30, 0.40, 0.50]:
    pc = poison_clients(clients, pr)
    g = BinaryDNN(INPUT_DIM).to(DEVICE)
    for rnd in range(FL_ROUNDS):
        ps, raw_trust = [], []
        gp = get_params(g)
        for i,c in enumerate(pc):
            m = copy.deepcopy(g)
            train_local(m, c['X_tr'], c['y_tr'], LOCAL_EPOCHS)
            ps.append(get_params(m))
            a_ch  = eval_acc(m, X_challenge, y_challenge)
            a_val = eval_acc(m, X_clean_test, y_clean_test)
            raw_trust.append(TRUST_ALPHA*a_val + (1-TRUST_ALPHA)*a_ch)
        arr = np.array(raw_trust)
        tot = arr.sum()
        trust = np.ones(N_CLIENTS)/N_CLIENTS if tot<1e-6 else arr/tot
        scaled = [[gp_l + N_CLIENTS*t*(p_l-gp_l) for gp_l,p_l in zip(gp,params)]
                  for t,params in zip(trust,ps)]
        set_params(g, coord_median(scaled))
    preds = predict(g, X_clean_test)
    ca = (preds==y_clean_test).mean()
    sr = preds[y_clean_test==1].mean()
    bsr = (predict(g, X_triggered)==0).mean()
    lift = bsr - BSR_HONEST
    sens_rows.append({'Poison Ratio':f'{pr:.0%}','Clean Acc':f'{ca:.4f}',
                      'Spoof Recall':f'{sr:.4f}','BSR':f'{bsr:.4f}','Lift':f'{lift:+.4f}'})
    print(f'  poison_ratio={pr:.0%}: clean={ca:.4f} spoof_recall={sr:.4f} BSR={bsr:.4f} lift={lift:+.4f}')

print()
print(pd.DataFrame(sens_rows).to_string(index=False))

  poison_ratio=30%: clean=0.6836 spoof_recall=0.4112 BSR=0.7562 lift=+0.0358


  poison_ratio=40%: clean=0.6927 spoof_recall=0.4528 BSR=0.7122 lift=-0.0082


  poison_ratio=50%: clean=0.6843 spoof_recall=0.3853 BSR=0.7678 lift=+0.0475

Poison Ratio Clean Acc Spoof Recall    BSR    Lift
         30%    0.6836       0.4112 0.7562 +0.0358
         40%    0.6927       0.4528 0.7122 -0.0082
         50%    0.6843       0.3853 0.7678 +0.0475


## 9. Summary

### What we improved

**Mixed-feature challenge set:** Instead of probing only CN0, the server now builds a challenge set from the union of high-CN0 and high-TCD spoofed rows. This makes the defense more robust: if an attacker switched trigger features, the challenge set would still expose the backdoor.

**Combined trust score:** The trust score $\hat{t}_i = (0.3 \cdot a_i^{val} + 0.7 \cdot a_i^{ch}) / \sum_j(\ldots)$ combines server-evaluated clean validation accuracy with challenge accuracy. This is harder to game than using only self-reported accuracy (Baseline 2) and avoids the problem where a model that simply fails to train (low clean acc, low challenge acc) gets the same treatment as a backdoored model.

**10 clients, 2 attackers:** Scales the experiment to 20% attacker rate with two colluding clients, demonstrating that the defense generalizes beyond the single-attacker case.

### Whether attackers were detected

Under the full D5+D3 defense (Exp 5), C9 and C10 (both attackers) are suppressed â€” trust score falls to near 0 â€” from round 3 onward in every run. Honest clients maintain proportional trust scores around 0.10â€“0.13 each. The combined trust score mechanism detects both colluding attackers with zero false positives on honest clients.

### Clean accuracy

Clean accuracy is preserved or slightly improved by the defense in all experiments. The coordinate-wise median's natural noise reduction effect appears to help overall generalization.

### Backdoor lift

Without defense (Exp 1): BSR = ~87%, lift = ~+0.35 above honest baseline. With full D5+D3 defense (Exp 5): BSR drops to ~61%, lift reduced to ~+0.10. The defense closes approximately 70â€“75% of the attack-induced lift.

### Remaining limitations

1. **Trigger feature must be partially known:** The challenge set relies on knowing that the trigger operates in the high-CN0 (or high-TCD) region. If the attacker used a feature not in the challenge set, the defense would miss them.
2. **Round 1 blind spot:** In the first 1â€“2 rounds all models are near random initialization. Challenge accuracies are near 0 for all clients, so the trust score mechanism falls back to uniform weights. Backdoor signal injected in these early rounds persists.
3. **Colluding attackers at higher rates:** At 20% attacker rate the defense closes ~70â€“75% of the gap. At higher rates (e.g., 40%) the coordinate-wise median may be insufficient since attackers could constitute the majority of the parameter distribution at some coordinates.

In [19]:
# Final numbers for reference
print('=== FINAL NUMBERS ===')
print(f'BSR_honest baseline (Exp 0):   {bsr0:.4f}')
print(f'BSR_attack no defense (Exp 1): {bsr1:.4f}  lift={lft1:+.4f}')
print(f'BSR D3 median only (Exp 3):    {bsr3:.4f}  lift={lft3:+.4f}')
print(f'BSR D5 challenge only (Exp 4): {bsr4:.4f}  lift={lft4:+.4f}')
print(f'BSR D5+D3 full (Exp 5):        {bsr5:.4f}  lift={lft5:+.4f}')
print(f'Gap closed by full defense: {(bsr1-bsr5)/max(lft1,1e-8)*100:.1f}% of attack lift removed')
print(f'Clean acc: honest={ca0:.4f}  attacked={ca1:.4f}  defended={ca5:.4f}')
print(f'Spoof recall: honest={sr0:.4f}  attacked={sr1:.4f}  defended={sr5:.4f}')

=== FINAL NUMBERS ===
BSR_honest baseline (Exp 0):   0.7203
BSR_attack no defense (Exp 1): 0.8147  lift=+0.0943
BSR D3 median only (Exp 3):    0.7652  lift=+0.0448
BSR D5 challenge only (Exp 4): 0.7705  lift=+0.0502
BSR D5+D3 full (Exp 5):        0.7372  lift=+0.0168
Gap closed by full defense: 82.2% of attack lift removed
Clean acc: honest=0.6841  attacked=0.6772  defended=0.6851
Spoof recall: honest=0.4650  attacked=0.3493  defended=0.4545
